# LeetCode 732 — My Calendar III bằng Python

- Link: https://leetcode.com/problems/my-calendar-iii/
- Độ khó: **Hard**
- Chủ đề: Thiết kế lớp, Quét sự kiện (sweep line), Mảng hiệu (difference array) trên `dict`, Tìm kiếm nhị phân (`bisect`), Segment Tree động với lazy propagation

Skeleton trên LeetCode (Python3):

```python
class MyCalendarThree:

    def __init__(self):

    def book(self, startTime: int, endTime: int) -> int:


# Your MyCalendarThree object will be instantiated and called as such:
# obj = MyCalendarThree()
# param_1 = obj.book(startTime,endTime)
```

## 1. Giới thiệu & mục tiêu học

Bạn quản lý một lịch hẹn. Mỗi lần thêm một sự kiện `[start, end)`, hệ thống phải trả lời: "**Hiện tại, tại một thời điểm nào đó, có tối đa bao nhiêu sự kiện chồng lên nhau?**" — gọi là *k-booking* lớn nhất.

Đây là bài kinh điển để học **quét sự kiện (sweep line)**: thay vì nhìn từng khoảng, ta nhìn từng **mốc thời gian** mà số sự kiện đang diễn ra thay đổi (+1 khi bắt đầu, −1 khi kết thúc).

Sau notebook này bạn sẽ:

1. Hiểu khoảng nửa mở `[start, end)` và vì sao hai sự kiện `[10,20)`, `[20,30)` **không** chồng nhau.
2. Viết brute force đếm chồng lấn tại từng điểm bắt đầu.
3. Dùng **mảng hiệu trên `dict`** + quét theo thứ tự để tìm số chồng lấn lớn nhất.
4. Duy trì danh sách mốc **đã sắp xếp** bằng `bisect.insort` để tránh sắp xếp lại.
5. Cài **Segment Tree động** (tạo nút khi cần) với thao tác "cộng 1 trên đoạn" và "max toàn cục" — kỹ thuật cho tọa độ tới `10^9`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Một *k-booking* xảy ra khi có `k` sự kiện cùng có một khoảng thời gian chung khác rỗng. Cho các sự kiện `[startTime, endTime)` (khoảng nửa mở: gồm mọi số thực `t` với `startTime <= t < endTime`), sau **mỗi** lần đặt lịch, trả về `k` lớn nhất sao cho tồn tại một k-booking trong lịch.

Cài đặt lớp `MyCalendarThree`:

- `MyCalendarThree()`: khởi tạo.
- `book(startTime, endTime)`: thêm sự kiện và trả về `k` lớn nhất hiện tại.

**Ví dụ**

```text
Input:
["MyCalendarThree", "book", "book", "book", "book", "book", "book"]
[[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]
Output:
[null, 1, 1, 2, 3, 3, 3]
Giải thích:
book(10, 20) -> 1  (chỉ một sự kiện)
book(50, 60) -> 1  (không chồng với [10,20))
book(10, 40) -> 2  ([10,20) và [10,40) chồng nhau)
book(5, 15)  -> 3  ([10,15) bị 3 sự kiện phủ)
book(5, 10)  -> 3  (tại [5,10) chỉ có 2 sự kiện, max vẫn là 3)
book(25, 55) -> 3  ([25,40) và [50,55) chỉ có 2)
```

**Ràng buộc**

- `0 <= startTime < endTime <= 10^9`
- Gọi `book` tối đa `400` lần.

In [1]:
def run(cls, ops, args):
    """Chạy chuỗi lệnh theo định dạng LeetCode, trả về list kết quả."""
    out, obj = [], None
    for op, a in zip(ops, args):
        if op == "MyCalendarThree":
            obj = cls(); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

EX_OPS = ["MyCalendarThree", "book", "book", "book", "book", "book", "book"]
EX_ARGS = [[], [10, 20], [50, 60], [10, 40], [5, 15], [5, 10], [25, 55]]
EX_OUT = [None, 1, 1, 2, 3, 3, 3]
print(list(zip(EX_ARGS, EX_OUT)))

[([], None), ([10, 20], 1), ([50, 60], 1), ([10, 40], 2), ([5, 15], 3), ([5, 10], 3), ([25, 55], 3)]


## 3. Kiến thức nền

### 3.1. Khoảng nửa mở và điều kiện chồng nhau

`[a, b)` và `[c, d)` chồng nhau (có phần chung độ dài dương) ⇔ `a < d and c < b`. Hai khoảng "chạm mép" như `[10,20)` và `[20,30)` **không** chồng nhau.

In [2]:
def overlap(a, b, c, d):
    return a < d and c < b

print(overlap(10, 20, 20, 30), overlap(10, 20, 15, 25), overlap(10, 20, 0, 11))

False True True


### 3.2. Mảng hiệu (difference array)

Muốn "cộng 1 cho mọi vị trí trong `[s, e)`", ta chỉ ghi `diff[s] += 1`, `diff[e] -= 1`. Sau đó **cộng dồn** (prefix sum) từ trái sang phải sẽ cho ra giá trị thật tại mỗi vị trí.

In [3]:
diff = [0] * 12
for s, e in [(2, 6), (4, 9), (5, 7)]:
    diff[s] += 1
    diff[e] -= 1
cur, values = 0, []
for x in diff:
    cur += x
    values.append(cur)
print("diff  :", diff)
print("values:", values, " max =", max(values))

diff  : [0, 0, 1, 0, 1, 1, -1, -1, 0, -1, 0, 0]
values: [0, 0, 1, 1, 2, 3, 2, 1, 1, 0, 0, 0]  max = 3


### 3.3. Mảng hiệu trên `dict` khi tọa độ lớn

Tọa độ tới `10^9` → không thể tạo mảng. Dùng `dict` chỉ lưu những mốc có thay đổi, rồi **duyệt các khoá theo thứ tự tăng** (`sorted`).

In [4]:
from collections import defaultdict
delta = defaultdict(int)
for s, e in [(10, 20), (10, 40), (5, 15)]:
    delta[s] += 1
    delta[e] -= 1
cur = best = 0
for t in sorted(delta):
    cur += delta[t]
    best = max(best, cur)
    print(f"t={t:>2}: delta={delta[t]:+d}, đang diễn ra={cur}")
print("max =", best)

t= 5: delta=+1, đang diễn ra=1
t=10: delta=+2, đang diễn ra=3
t=15: delta=-1, đang diễn ra=2
t=20: delta=-1, đang diễn ra=1
t=40: delta=-1, đang diễn ra=0
max = 3


### 3.4. `bisect.insort` — chèn giữ thứ tự

`insort(a, x)` chèn `x` vào list đã sắp xếp `a` sao cho vẫn sắp xếp. Tìm vị trí `O(log n)`, nhưng dịch phần tử `O(n)` — vẫn rất nhanh với `n` nhỏ (≤ 800 mốc).

In [5]:
from bisect import insort, bisect_left
a = [5, 10, 20]
insort(a, 15); insort(a, 1)
print(a, bisect_left(a, 15))

[1, 5, 10, 15, 20] 3


### 3.5. Lớp nút có `__slots__` cho cây động

In [6]:
class Node:
    __slots__ = ("left", "right", "mx", "lazy")
    def __init__(self):
        self.left = self.right = None
        self.mx = 0       # max trên đoạn của nút
        self.lazy = 0     # giá trị cộng "treo", chưa đẩy xuống con

nd = Node()
print(nd.mx, nd.lazy, nd.left)

0 0 None


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Max chồng lấn chỉ đổi tại các mốc

Hàm `cnt(t)` = số sự kiện chứa thời điểm `t` là **hàm bậc thang**: chỉ thay đổi tại các `start` (tăng 1) và các `end` (giảm 1). Vậy giá trị lớn nhất của `cnt` đạt được tại một mốc `start` nào đó.

### 4.2. Tính tay ví dụ (sau 4 lần đặt)

Các sự kiện: `[10,20)`, `[50,60)`, `[10,40)`, `[5,15)`.

| Mốc t | Δ | cnt sau mốc |
|---|---|---|
| 5 | +1 | 1 |
| 10 | +2 | 3 |
| 15 | −1 | 2 |
| 20 | −1 | 1 |
| 40 | −1 | 0 |
| 50 | +1 | 1 |
| 60 | −1 | 0 |

Max = 3 (trên `[10, 15)`). ✔

### 4.3. Cẩn thận khi `start` của sự kiện này trùng `end` của sự kiện khác

Tại cùng một mốc `t`, phải **cộng hết Δ của mốc đó** rồi mới đo max. Nếu xử lý `+1` trước `-1` riêng lẻ, ta sẽ đếm nhầm `[10,20)` và `[20,30)` là chồng nhau. Gộp Δ theo khoá `dict` tự động tránh lỗi này.

In [7]:
delta = defaultdict(int)
for s, e in [(10, 20), (20, 30)]:
    delta[s] += 1; delta[e] -= 1
print(dict(sorted(delta.items())), "-> tại t=20 Δ =", delta[20], "nên không bao giờ đạt 2")

{10: 1, 20: 0, 30: -1} -> tại t=20 Δ = 0 nên không bao giờ đạt 2


## 5. Cách 1 — Brute Force: đếm tại mọi điểm bắt đầu

**Ý tưởng.** Lưu mọi sự kiện. Sau mỗi `book`, với mỗi sự kiện `[s, e)`, đếm số sự kiện chứa thời điểm `s` (tức `s2 <= s < e2`). Max chồng lấn = max các con số đó (theo 4.1).

In [8]:
class MyCalendarThreeBrute:
    def __init__(self):
        self.events = []

    def book(self, startTime, endTime):
        self.events.append((startTime, endTime))
        best = 0
        for s, _ in self.events:
            c = sum(1 for s2, e2 in self.events if s2 <= s < e2)
            best = max(best, c)
        return best

print(run(MyCalendarThreeBrute, EX_OPS, EX_ARGS))

[None, 1, 1, 2, 3, 3, 3]


**Độ phức tạp.** Lần gọi thứ `n` tốn `O(n^2)` → tổng `O(N^3)` với `N = 400` là `6.4·10^7` — chậm trong Python nhưng làm **trọng tài** tốt.

**Nút thắt.** Mỗi lần ta tính lại từ đầu số chồng lấn tại mọi điểm. Thay vào đó, chỉ cần một lần **quét** qua các mốc đã sắp xếp với cộng dồn (`O(n)`), hoặc tốt hơn nữa: một cấu trúc dữ liệu cập nhật được **cục bộ** (segment tree).

## 6. Bước đột phá

**Insight 1 (sweep line).** Biến mỗi sự kiện thành hai "sự kiện điểm": `(start, +1)` và `(end, −1)`. Gộp theo thời điểm, duyệt theo thứ tự tăng, cộng dồn → `cnt` sau mỗi mốc. Max của các giá trị cộng dồn là đáp án.

**Bất biến.** Sau khi xử lý mọi mốc `≤ t`, biến cộng dồn `cur` = số sự kiện chứa khoảng `[t, mốc_kế_tiếp)`.

**Insight 2 (cập nhật cục bộ).** Thêm sự kiện `[s, e)` = **cộng 1 trên đoạn** `[s, e)` của hàm `cnt`. Câu hỏi = **max toàn cục** của `cnt`. Đây chính là bài toán *range add + range max* → **Segment Tree với lazy propagation**. Vì tọa độ tới `10^9`, ta tạo nút **động** (chỉ khi cần), mỗi lần cập nhật tạo tối đa `O(log C)` nút mới.

**Quy tắc nếu… thì… (segment tree):**

- Nếu đoạn của nút **nằm ngoài** `[s, e)` → không làm gì.
- Nếu đoạn của nút **nằm trọn** trong `[s, e)` → `mx += 1`, `lazy += 1`, dừng.
- Ngược lại → đi xuống hai con, rồi `mx = lazy + max(mx_trái, mx_phải)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mảng hiệu trên `dict` + sắp xếp mỗi lần

**Giải thích.** Giữ `delta: dict[t] -> tổng Δ`. Mỗi `book`: cập nhật hai khoá rồi quét `sorted(delta)` để lấy max cộng dồn.

In [9]:
class MyCalendarThreeDiff:
    def __init__(self):
        self.delta = defaultdict(int)

    def book(self, startTime, endTime):
        self.delta[startTime] += 1
        self.delta[endTime] -= 1
        cur = best = 0
        for t in sorted(self.delta):
            cur += self.delta[t]
            if cur > best:
                best = cur
        return best

print(run(MyCalendarThreeDiff, EX_OPS, EX_ARGS))

[None, 1, 1, 2, 3, 3, 3]


**Code nộp LeetCode (Cách 2).**

```python
from collections import defaultdict

class MyCalendarThree:

    def __init__(self):
        self.delta = defaultdict(int)

    def book(self, startTime: int, endTime: int) -> int:
        self.delta[startTime] += 1
        self.delta[endTime] -= 1
        cur = best = 0
        for t in sorted(self.delta):
            cur += self.delta[t]
            best = max(best, cur)
        return best
```

**Độ phức tạp.** Mỗi `book`: `O(n log n)` (sắp xếp `≤ 2n` khoá). Tổng `O(N^2 log N)` ≈ `400 · 800 · 10` — rất nhanh. Space `O(N)`.

### 7.2. Cách 3 — Danh sách mốc đã sắp xếp + `insort`

**Giải thích.** Tránh sắp xếp lại mỗi lần: giữ `keys` là list các mốc **đã sắp xếp**; khi gặp mốc mới thì `insort`. Quét `keys` theo thứ tự để cộng dồn. Mỗi `book` chỉ còn `O(n)`.

Tối ưu thêm: max chỉ có thể **tăng** trong đoạn `[start, end)` vừa thêm, nên thực ra chỉ cần quét lại tới `end`. Nhưng để giữ code đơn giản ta quét toàn bộ và giữ `best` toàn cục là max của mọi lần (max không bao giờ giảm vì không có xoá).

In [10]:
class MyCalendarThreeSorted:
    def __init__(self):
        self.keys = []          # các mốc đã sắp xếp
        self.delta = {}
        self.best = 0

    def _add(self, t, d):
        if t not in self.delta:
            insort(self.keys, t)
            self.delta[t] = 0
        self.delta[t] += d

    def book(self, startTime, endTime):
        self._add(startTime, 1)
        self._add(endTime, -1)
        cur = 0
        for t in self.keys:
            cur += self.delta[t]
            if t >= endTime:    # sau end không thể đạt giá trị mới lớn hơn do sự kiện này
                break
            if cur > self.best:
                self.best = cur
        return self.best

print(run(MyCalendarThreeSorted, EX_OPS, EX_ARGS))

[None, 1, 1, 2, 3, 3, 3]


> Vì sao được dừng ở `endTime`? Trước lần `book` này, `best` đã là max cũ. Sự kiện mới chỉ làm tăng `cnt` trên `[start, end)`, nên giá trị mới lớn hơn `best` (nếu có) chỉ xuất hiện tại các mốc `< end`.

**Code nộp LeetCode (Cách 3).**

```python
from bisect import insort

class MyCalendarThree:

    def __init__(self):
        self.keys = []
        self.delta = {}
        self.best = 0

    def _add(self, t, d):
        if t not in self.delta:
            insort(self.keys, t)
            self.delta[t] = 0
        self.delta[t] += d

    def book(self, startTime: int, endTime: int) -> int:
        self._add(startTime, 1)
        self._add(endTime, -1)
        cur = 0
        for t in self.keys:
            cur += self.delta[t]
            if t >= endTime:
                break
            if cur > self.best:
                self.best = cur
        return self.best
```

**Độ phức tạp.** Mỗi `book`: `O(n)` (insort + quét). Tổng `O(N^2)`. Space `O(N)`.

### 7.3. Cách 4 — Segment Tree động với lazy propagation

**Giải thích.** Cây phủ đoạn `[0, 10^9]` (đủ cho mọi `start`; ta cập nhật đoạn đóng `[start, end-1]` vì khoảng nửa mở trên số nguyên). Mỗi nút giữ:

- `mx`: max của `cnt` trên đoạn của nút (đã tính cả `lazy` của chính nó),
- `lazy`: lượng đã cộng cho **cả đoạn** mà chưa đẩy xuống con.

Ở đây ta dùng kỹ thuật **không đẩy lazy xuống** (lazy "dính" tại nút): `mx = lazy + max(mx_trái, mx_phải)`. Đáp án = `root.mx`.

In [11]:
class MyCalendarThreeSegTree:
    LO, HI = 0, 10**9

    def __init__(self):
        self.root = Node()

    def _update(self, node, lo, hi, l, r):
        if r < lo or hi < l:
            return
        if l <= lo and hi <= r:
            node.mx += 1
            node.lazy += 1
            return
        mid = (lo + hi) // 2
        if node.left is None:
            node.left = Node()
        if node.right is None:
            node.right = Node()
        self._update(node.left, lo, mid, l, r)
        self._update(node.right, mid + 1, hi, l, r)
        node.mx = node.lazy + max(node.left.mx, node.right.mx)

    def book(self, startTime, endTime):
        self._update(self.root, self.LO, self.HI, startTime, endTime - 1)
        return self.root.mx

print(run(MyCalendarThreeSegTree, EX_OPS, EX_ARGS))

[None, 1, 1, 2, 3, 3, 3]


**Code nộp LeetCode (Cách 4).**

```python
class Node:
    __slots__ = ('left', 'right', 'mx', 'lazy')
    def __init__(self):
        self.left = self.right = None
        self.mx = 0
        self.lazy = 0

class MyCalendarThree:

    def __init__(self):
        self.root = Node()

    def _update(self, node, lo, hi, l, r):
        if r < lo or hi < l:
            return
        if l <= lo and hi <= r:
            node.mx += 1
            node.lazy += 1
            return
        mid = (lo + hi) // 2
        if node.left is None:
            node.left = Node()
        if node.right is None:
            node.right = Node()
        self._update(node.left, lo, mid, l, r)
        self._update(node.right, mid + 1, hi, l, r)
        node.mx = node.lazy + max(node.left.mx, node.right.mx)

    def book(self, startTime: int, endTime: int) -> int:
        self._update(self.root, 0, 10**9, startTime, endTime - 1)
        return self.root.mx
```

**Độ phức tạp.** Mỗi `book`: `O(log C)` với `C = 10^9` (~30 tầng, mỗi tầng ≤ 4 nút được thăm). Space `O(N log C)` nút. Đây là cách **mở rộng tốt nhất** nếu số lần gọi lên `10^5`.

### 7.4. (Bonus) Đếm số nút cây đã tạo

Mỗi lần cập nhật tạo tối đa khoảng `4 · 30` nút. Kiểm tra thực nghiệm:

In [12]:
def count_nodes(node):
    if node is None:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)

import random
rng = random.Random(1)
cal = MyCalendarThreeSegTree()
for _ in range(400):
    s = rng.randint(0, 10**9 - 1); e = rng.randint(s + 1, 10**9)
    cal.book(s, e)
print("Số nút sau 400 lần book:", count_nodes(cal.root), " (giới hạn thô 400*4*31 =", 400 * 4 * 31, ")")

Số nút sau 400 lần book: 30931  (giới hạn thô 400*4*31 = 49600 )


## 8. TRACE từng bước

### 8.1. Bảng trạng thái Cách 2 (mảng hiệu) qua 6 lần `book`

| Lần | Sự kiện | delta (sắp xếp) | Cộng dồn | Trả về |
|---|---|---|---|---|
| 1 | [10,20) | 10:+1, 20:−1 | 1, 0 | 1 |
| 2 | [50,60) | 10:+1, 20:−1, 50:+1, 60:−1 | 1,0,1,0 | 1 |
| 3 | [10,40) | 10:+2, 20:−1, 40:−1, 50:+1, 60:−1 | 2,1,0,1,0 | 2 |
| 4 | [5,15) | 5:+1, 10:+2, 15:−1, 20:−1, 40:−1, ... | 1,3,2,1,0,... | 3 |
| 5 | [5,10) | 5:+2, 10:+1, 15:−1, ... | 2,3,2,... | 3 |
| 6 | [25,55) | ..., 25:+1, 40:−1, 50:+1, 55:−1, 60:−1 | ...,2,1,2,1,0 | 3 |

In [13]:
def trace_diff(ops_args):
    delta = defaultdict(int)
    for s, e in ops_args:
        delta[s] += 1; delta[e] -= 1
        cur = best = 0; prefix = []
        for t in sorted(delta):
            cur += delta[t]; prefix.append(cur); best = max(best, cur)
        items = ", ".join(f"{t}:{delta[t]:+d}" for t in sorted(delta) if delta[t])
        print(f"book({s},{e}) -> {best} | delta {{{items}}} | cộng dồn {prefix}")

trace_diff(EX_ARGS[1:])

book(10,20) -> 1 | delta {10:+1, 20:-1} | cộng dồn [1, 0]
book(50,60) -> 1 | delta {10:+1, 20:-1, 50:+1, 60:-1} | cộng dồn [1, 0, 1, 0]
book(10,40) -> 2 | delta {10:+2, 20:-1, 40:-1, 50:+1, 60:-1} | cộng dồn [2, 1, 0, 1, 0]
book(5,15) -> 3 | delta {5:+1, 10:+2, 15:-1, 20:-1, 40:-1, 50:+1, 60:-1} | cộng dồn [1, 3, 2, 1, 0, 1, 0]
book(5,10) -> 3 | delta {5:+2, 10:+1, 15:-1, 20:-1, 40:-1, 50:+1, 60:-1} | cộng dồn [2, 3, 2, 1, 0, 1, 0]
book(25,55) -> 3 | delta {5:+2, 10:+1, 15:-1, 20:-1, 25:+1, 40:-1, 50:+1, 55:-1, 60:-1} | cộng dồn [2, 3, 2, 1, 2, 1, 2, 1, 0]


### 8.2. Trace segment tree trên miền nhỏ `[0, 7]`

Để dễ nhìn, thu nhỏ miền thành `[0, 7]` và đặt `[1,5)`, `[3,8)`, `[2,4)`. In các nút được thăm.

In [14]:
def trace_seg(bookings, LO=0, HI=7):
    root = Node()
    def upd(node, lo, hi, l, r, depth):
        if r < lo or hi < l:
            return
        pad = "  " * depth
        if l <= lo and hi <= r:
            node.mx += 1; node.lazy += 1
            print(f"{pad}[{lo},{hi}] nằm trọn -> mx={node.mx}, lazy={node.lazy}")
            return
        mid = (lo + hi) // 2
        node.left = node.left or Node(); node.right = node.right or Node()
        print(f"{pad}[{lo},{hi}] cắt ngang, đi xuống")
        upd(node.left, lo, mid, l, r, depth + 1)
        upd(node.right, mid + 1, hi, l, r, depth + 1)
        node.mx = node.lazy + max(node.left.mx, node.right.mx)
        print(f"{pad}[{lo},{hi}] cập nhật mx = {node.lazy} + max({node.left.mx},{node.right.mx}) = {node.mx}")
    for s, e in bookings:
        print(f"--- book({s},{e}) = cộng 1 trên [{s},{e-1}]")
        upd(root, LO, HI, s, e - 1, 0)
        print(f"=> max toàn cục = {root.mx}")

trace_seg([(1, 5), (3, 8), (2, 4)])

--- book(1,5) = cộng 1 trên [1,4]
[0,7] cắt ngang, đi xuống
  [0,3] cắt ngang, đi xuống
    [0,1] cắt ngang, đi xuống
      [1,1] nằm trọn -> mx=1, lazy=1
    [0,1] cập nhật mx = 0 + max(0,1) = 1
    [2,3] nằm trọn -> mx=1, lazy=1
  [0,3] cập nhật mx = 0 + max(1,1) = 1
  [4,7] cắt ngang, đi xuống
    [4,5] cắt ngang, đi xuống
      [4,4] nằm trọn -> mx=1, lazy=1
    [4,5] cập nhật mx = 0 + max(1,0) = 1
  [4,7] cập nhật mx = 0 + max(1,0) = 1
[0,7] cập nhật mx = 0 + max(1,1) = 1
=> max toàn cục = 1
--- book(3,8) = cộng 1 trên [3,7]
[0,7] cắt ngang, đi xuống
  [0,3] cắt ngang, đi xuống
    [2,3] cắt ngang, đi xuống
      [3,3] nằm trọn -> mx=1, lazy=1
    [2,3] cập nhật mx = 1 + max(0,1) = 2
  [0,3] cập nhật mx = 0 + max(1,2) = 2
  [4,7] nằm trọn -> mx=2, lazy=1
[0,7] cập nhật mx = 0 + max(2,2) = 2
=> max toàn cục = 2
--- book(2,4) = cộng 1 trên [2,3]
[0,7] cắt ngang, đi xuống
  [0,3] cắt ngang, đi xuống
    [2,3] nằm trọn -> mx=3, lazy=2
  [0,3] cập nhật mx = 0 + max(1,3) = 3
[0,7] cập n

Kiểm tra bằng tay: `[1,5)`, `[3,8)`, `[2,4)` cùng chứa điểm 3 → max = 3. ✔

## 9. Code nộp LeetCode

Bản chính: **Cách 4 (Segment Tree động)** — `O(log C)` mỗi lần, mở rộng tốt. (Cách 2/3 cũng qua thoải mái với 400 lần gọi.)

In [15]:
class MyCalendarThree:

    def __init__(self):
        self.root = Node()

    def _update(self, node, lo, hi, l, r):
        if r < lo or hi < l:
            return
        if l <= lo and hi <= r:
            node.mx += 1
            node.lazy += 1
            return
        mid = (lo + hi) // 2
        if node.left is None:
            node.left = Node()
        if node.right is None:
            node.right = Node()
        self._update(node.left, lo, mid, l, r)
        self._update(node.right, mid + 1, hi, l, r)
        node.mx = node.lazy + max(node.left.mx, node.right.mx)

    def book(self, startTime: int, endTime: int) -> int:
        self._update(self.root, 0, 10**9, startTime, endTime - 1)
        return self.root.mx

assert run(MyCalendarThree, EX_OPS, EX_ARGS) == EX_OUT
print("MyCalendarThree OK trên ví dụ đề")

MyCalendarThree OK trên ví dụ đề


## 10. Vì sao thuật toán đúng

**Sweep line (Cách 2, 3).** Với tọa độ nguyên và khoảng nửa mở, `cnt(t) = Σ_{mốc u ≤ t} delta[u]` (định nghĩa mảng hiệu). `cnt` là hằng số trên mỗi khoảng giữa hai mốc liên tiếp, nên max của `cnt` trên toàn trục = max của các tổng cộng dồn tại các mốc. Gộp Δ theo khoá đảm bảo tại mốc `t` vừa là `end` vừa là `start`, giá trị đo được là `cnt` **thật** trên `[t, mốc kế)` — không có "đỉnh giả".

**Dừng sớm ở Cách 3.** Gọi `best_cũ` là max trước lần `book`. Sau `book`, `cnt_mới(t) = cnt_cũ(t) + [s ≤ t < e]`. Với `t ≥ e`, `cnt_mới(t) = cnt_cũ(t) ≤ best_cũ`. Vậy chỉ cần xét các mốc `< e`.

**Segment tree (Cách 4).** Bất biến tại mỗi nút đoạn `[lo, hi]`: `mx = (tổng lazy trên đường từ nút này xuống) + ...` chính xác là: `mx` = max trên `[lo, hi]` của số lần cộng **được ghi tại nút này hoặc tại con cháu của nó**. Khi đoạn nằm trọn, cộng 1 vào cả đoạn làm max tăng đúng 1 → `mx += 1`, ghi `lazy += 1`. Khi cắt ngang, sau khi cập nhật con, `mx = lazy + max(con)` khôi phục bất biến (các lần cộng ghi tại nút áp dụng cho toàn đoạn, cộng thêm vào max của con). Tại gốc, không có tổ tiên → `root.mx` = max toàn cục thật.

**Con chưa tạo** tương đương với nút `mx = 0, lazy = 0` — đúng vì chưa có cộng nào ghi vào đó.

## 11. Phân tích Time/Space

`N` = số lần `book` (≤ 400), `C = 10^9`.

| Cách | Mỗi `book` | Tổng | Space |
|---|---|---|---|
| 1. Brute | `O(n^2)` | `O(N^3)` | `O(N)` |
| 2. Dict + sort | `O(n log n)` | `O(N^2 log N)` | `O(N)` |
| 3. insort + quét đến end | `O(n)` | `O(N^2)` | `O(N)` |
| 4. Segment tree động | `O(log C)` | `O(N log C)` | `O(N log C)` |

Thử với `N = 1500` (vượt đề) để thấy sự khác biệt:

In [16]:
import time
rng = random.Random(732)
calls = []
for _ in range(1500):
    s = rng.randint(0, 10**9 - 1); calls.append((s, rng.randint(s + 1, min(10**9, s + 10**8))))
for cls in [MyCalendarThreeDiff, MyCalendarThreeSorted, MyCalendarThree]:
    obj = cls(); t = time.perf_counter()
    res = [obj.book(s, e) for s, e in calls]
    print(f"{cls.__name__:<24} {(time.perf_counter() - t)*1000:8.1f} ms  max cuối = {res[-1]}")

MyCalendarThreeDiff         268.5 ms  max cuối = 97
MyCalendarThreeSorted        48.7 ms  max cuối = 97
MyCalendarThree              34.2 ms  max cuối = 97


## 12. Bộ test

In [17]:
ALL = [MyCalendarThreeBrute, MyCalendarThreeDiff, MyCalendarThreeSorted, MyCalendarThreeSegTree, MyCalendarThree]

for cls in ALL:
    assert run(cls, EX_OPS, EX_ARGS) == EX_OUT, cls.__name__

# Edge cases
EDGE = [
    ([(0, 1)], [1]),
    ([(0, 10**9)], [1]),
    ([(10, 20), (20, 30)], [1, 1]),                 # chạm mép không chồng
    ([(10, 20), (19, 30)], [1, 2]),
    ([(0, 10**9)] * 5, [1, 2, 3, 4, 5]),            # trùng hoàn toàn
    ([(1, 2), (3, 4), (5, 6)], [1, 1, 1]),
    ([(0, 10**9), (10**9 - 1, 10**9)], [1, 2]),     # sát biên phải
]
for cls in ALL:
    for books, exp in EDGE:
        obj = cls()
        assert [obj.book(s, e) for s, e in books] == exp, (cls.__name__, books)
print("Ví dụ đề + edge case OK")

Ví dụ đề + edge case OK


In [18]:
rng = random.Random(7320)
for _ in range(300):
    objs = [cls() for cls in ALL]
    for _ in range(rng.randint(1, 30)):
        s = rng.randint(0, 30); e = rng.randint(s + 1, 31)
        res = [o.book(s, e) for o in objs]
        assert len(set(res)) == 1, (res, s, e)
print("300 chuỗi đặt lịch ngẫu nhiên (tọa độ nhỏ): mọi cách khớp brute force")

300 chuỗi đặt lịch ngẫu nhiên (tọa độ nhỏ): mọi cách khớp brute force


In [19]:
# Tọa độ lớn, đối chiếu brute force với số lần gọi vừa phải
for _ in range(20):
    objs = [cls() for cls in ALL]
    for _ in range(60):
        s = rng.randint(0, 10**9 - 1); e = rng.randint(s + 1, 10**9)
        res = [o.book(s, e) for o in objs]
        assert len(set(res)) == 1
print("Tọa độ tới 10^9: khớp")

Tọa độ tới 10^9: khớp


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Coi khoảng là đóng `[start, end]`

**Code sai:** cập nhật segment tree trên `[start, end]` (thay vì `end - 1`), hoặc đếm chồng bằng `s2 <= s <= e2`.

**Vì sao sai:** `[10,20)` và `[20,30)` bị tính là 2-booking.

**Code đúng:** cập nhật `[start, end - 1]` trên số nguyên / dùng `<` cho đầu mút phải.

In [20]:
class SegClosedWrong(MyCalendarThree):
    def book(self, startTime, endTime):
        self._update(self.root, 0, 10**9, startTime, endTime)   # SAI: thiếu -1
        return self.root.mx

w, r = SegClosedWrong(), MyCalendarThree()
print("sai:", [w.book(10, 20), w.book(20, 30)], " đúng:", [r.book(10, 20), r.book(20, 30)])

sai: [1, 2]  đúng: [1, 1]


### Lỗi 2 — Quét sự kiện điểm mà xử lý `+1` trước `-1` tại cùng thời điểm

**Code sai:** tạo list sự kiện điểm với khoá phụ làm cho **start xếp trước end** tại cùng thời điểm, ví dụ `(t, 0, +1)` cho start và `(t, 1, -1)` cho end. (Nếu dùng đúng `(t, +1)`/`(t, -1)` thì `sort()` xếp `-1` trước — may mắn đúng; nhưng đừng dựa vào may mắn.)

**Vì sao sai:** tại `t = 20`, `[20,30)` được cộng trước khi `[10,20)` bị trừ → đỉnh giả bằng 2.

**Code đúng:** gộp Δ theo thời điểm (`dict`) hoặc chắc chắn `-1` được xử lý trước `+1`.

In [21]:
def sweep_wrong(events):
    pts = []
    for s, e in events:
        pts.append((s, 0, +1))   # start xếp TRƯỚC end tại cùng t -> SAI
        pts.append((e, 1, -1))
    cur = best = 0
    for _, _, d in sorted(pts):
        cur += d; best = max(best, cur)
    return best

print("sai:", sweep_wrong([(10, 20), (20, 30)]), " đúng: 1")

sai: 2  đúng: 1


### Lỗi 3 — Segment tree quên cộng `lazy` khi tính lại `mx`

**Code sai:** `node.mx = max(node.left.mx, node.right.mx)` sau khi cập nhật con.

**Vì sao sai:** những lần cộng "nằm trọn" trước đó được ghi tại `node.lazy`, không xuống con → bị mất.

**Code đúng:** `node.mx = node.lazy + max(...)`.

In [22]:
class SegNoLazyWrong(MyCalendarThree):
    def _update(self, node, lo, hi, l, r):
        if r < lo or hi < l:
            return
        if l <= lo and hi <= r:
            node.mx += 1; node.lazy += 1; return
        mid = (lo + hi) // 2
        node.left = node.left or Node(); node.right = node.right or Node()
        self._update(node.left, lo, mid, l, r)
        self._update(node.right, mid + 1, hi, l, r)
        node.mx = max(node.left.mx, node.right.mx)               # SAI

w, r = SegNoLazyWrong(), MyCalendarThree()
print("sai:", [w.book(0, 10**9 + 1), w.book(5, 6)], " đúng:", [r.book(0, 10**9 + 1), r.book(5, 6)])

sai: [1, 1]  đúng: [1, 2]


### Lỗi 4 — Trả về max của **lần đặt hiện tại** thay vì max toàn cục

**Code sai:** chỉ đếm số sự kiện chồng với sự kiện vừa thêm.

**Vì sao sai:** ví dụ đề lần 5 (`[5,10)`) chỉ chồng với 1 sự kiện, nhưng đáp án vẫn là 3 (từ trước).

**Code đúng:** max **toàn bộ lịch**.

In [23]:
class LocalWrong:
    def __init__(self): self.ev = []
    def book(self, s, e):
        self.ev.append((s, e))
        delta = defaultdict(int)
        for s2, e2 in self.ev:
            if s2 < e and s < e2:                         # chỉ sự kiện chồng với cái mới
                delta[max(s, s2)] += 1; delta[min(e, e2)] -= 1
        cur = best = 0
        for t in sorted(delta):
            cur += delta[t]; best = max(best, cur)
        return best

print("sai:", run(LocalWrong, EX_OPS, EX_ARGS), " đúng:", EX_OUT)

sai: [None, 1, 1, 2, 3, 2, 2]  đúng: [None, 1, 1, 2, 3, 3, 3]


## 14. Mẹo & nhận dạng pattern

- **"Tối đa bao nhiêu khoảng chồng nhau"** → sweep line (+1 tại start, −1 tại end). Cùng họ: *Meeting Rooms II (253)*, *Car Pooling (1094)*, *Minimum Number of Arrows (452)*.
- **Tọa độ lớn + ít mốc** → mảng hiệu trên `dict`/`SortedDict` hoặc nén tọa độ.
- **Cập nhật đoạn + truy vấn max/tổng, trực tuyến, tọa độ lớn** → Segment Tree **động** (tạo nút khi cần).
- **Khoảng nửa mở** → trên số nguyên, `[s, e)` = `[s, e-1]`.
- Lazy "dính" (không đẩy xuống) đủ dùng khi chỉ có **cộng** và chỉ truy vấn **toàn cục**.

## 15. Biến thể

1. **My Calendar I (729)**: từ chối sự kiện nếu tạo ra 2-booking.
2. **My Calendar II (731)**: từ chối nếu tạo ra 3-booking.
3. **Meeting Rooms II (253)**: số phòng tối thiểu = max chồng lấn (offline, sắp xếp một lần).
4. **Truy vấn max trên đoạn `[a, b)` bất kỳ** (không chỉ toàn cục): cần hàm `query` trên segment tree.
5. **Có thao tác huỷ sự kiện**: segment tree cộng `-1`; cách "best không giảm" ở Cách 3 không còn đúng.

In [24]:
# Biến thể 2: My Calendar II dùng chính segment tree của ta
class MyCalendarTwo:
    def __init__(self):
        self.tree = MyCalendarThree()
        self.events = []

    def book(self, start, end):
        # thử thêm vào bản sao nhẹ: dùng brute vì N nhỏ
        cnt = defaultdict(int)
        for s, e in self.events + [(start, end)]:
            cnt[s] += 1; cnt[e] -= 1
        cur = 0
        for t in sorted(cnt):
            cur += cnt[t]
            if cur >= 3:
                return False
        self.events.append((start, end))
        self.tree.book(start, end)
        return True

c2 = MyCalendarTwo()
res2 = [c2.book(*x) for x in [(10, 20), (50, 60), (10, 40), (5, 15), (5, 10), (25, 55)]]
assert res2 == [True, True, True, False, True, True]
print(res2)

[True, True, True, False, True, True]


Kết quả mong đợi của LeetCode 731 cho chuỗi trên là `[True, True, True, False, True, True]` (`[5,15)` bị từ chối vì `[10,15)` sẽ có 3 sự kiện; `[5,10)` thì được nhận). ✔

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time mỗi `book` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Đếm tại mỗi start | `O(n^2)` | `O(n)` | Trọng tài |
| 2. Dict + sort | Mảng hiệu, sắp xếp khoá, cộng dồn | `O(n log n)` | `O(n)` | Code ngắn nhất, N nhỏ |
| 3. insort + quét đến end | Giữ khoá sắp xếp, dừng sớm | `O(n)` | `O(n)` | N vừa, không muốn cây |
| 4. Segment tree động | Range add + global max | `O(log C)` | `O(N log C)` | N lớn, cần mở rộng |

## 17. Bài tập

**Bài 1 (đoán output).** Chuỗi `book(1,5)`, `book(2,3)`, `book(3,4)`, `book(2,4)` trả về gì?

**Bài 2 (tính tay).** Lập bảng mảng hiệu (như 8.1) cho `book(0,10)`, `book(5,15)`, `book(10,20)`, `book(5,10)`.

**Bài 3 (điền TODO).** Hoàn thiện `max_overlap(events)` dùng mảng hiệu trên `dict` (offline, một lần).

**Bài 4 (tự viết lại).** Viết lại lớp segment tree động `MySegCalendar` mà không nhìn code.

**Bài 5 (mini challenge).** Viết `min_rooms(intervals)` (Meeting Rooms II) dùng heap: sắp xếp theo start, heap lưu end của các phòng đang dùng. Đối chiếu với `max_overlap`.

In [25]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def max_overlap(events):
    # TODO: delta[s] += 1; delta[e] -= 1; duyệt sorted(delta) cộng dồn, lấy max
    pass

# Bài 4 — TODO
class MySegCalendar:
    def __init__(self):
        # TODO: self.root = Node()
        pass
    def book(self, startTime, endTime):
        # TODO: cập nhật [startTime, endTime - 1] trên [0, 10**9]; trả root.mx
        pass

# Bài 5 — TODO
def min_rooms(intervals):
    # TODO: sort theo start; heap end; nếu heap[0] <= start thì heappop; heappush(end)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `[1,5)` chứa mọi sự kiện sau.
- Mức 2: sau 3 lần: điểm 2 có 2, điểm 3 có 2 → max 2.
- Mức 3: `book(2,4)`: điểm 3 có `[1,5),[3,4),[2,4)` = 3 (điểm 2 cũng có 3) → `[1, 2, 2, 3]`.

**Bài 2.**
- Mức 1: lần 1: `{0:+1, 10:−1}` → 1.
- Mức 2: lần 2: `{0:+1, 5:+1, 10:−1, 15:−1}` → 2; lần 3: `10` có Δ = 0 → vẫn 2.
- Mức 3: lần 4: `0:+1, 5:+2, 10:−1, 15:−1, 20:−1` → cộng dồn `1, 3, 2, 1, 0` → **3**.

**Bài 3.**
- Mức 1: `defaultdict(int)`.
- Mức 2: duyệt khoá đã sắp xếp.
- Mức 3: cập nhật `best` sau khi cộng Δ của mốc.

**Bài 4.**
- Mức 1: 3 trường hợp: ngoài, nằm trọn, cắt ngang.
- Mức 2: nằm trọn: `mx += 1; lazy += 1`.
- Mức 3: cắt ngang: tạo con nếu thiếu, đệ quy, `mx = lazy + max(con)`.

**Bài 5.**
- Mức 1: `heapq`; sắp xếp theo start.
- Mức 2: phòng có `end <= start` hiện tại thì giải phóng.
- Mức 3: đáp án = kích thước heap lớn nhất (hoặc kích thước cuối nếu chỉ pop một lần mỗi bước).

## 19. Đáp án tham khảo

In [26]:
# Bài 1, 2
c = MyCalendarThree()
assert [c.book(1, 5), c.book(2, 3), c.book(3, 4), c.book(2, 4)] == [1, 2, 2, 3]
c = MyCalendarThree()
assert [c.book(0, 10), c.book(5, 15), c.book(10, 20), c.book(5, 10)] == [1, 2, 2, 3]

# Bài 3
def max_overlap_ans(events):
    delta = defaultdict(int)
    for s, e in events:
        delta[s] += 1; delta[e] -= 1
    cur = best = 0
    for t in sorted(delta):
        cur += delta[t]; best = max(best, cur)
    return best

assert max_overlap_ans([(10, 20), (50, 60), (10, 40), (5, 15)]) == 3
print("Bài 1–3 OK")

Bài 1–3 OK


In [27]:
# Bài 4
class MySegCalendarAns:
    def __init__(self):
        self.root = Node()
    def _upd(self, nd, lo, hi, l, r):
        if r < lo or hi < l:
            return
        if l <= lo and hi <= r:
            nd.mx += 1; nd.lazy += 1; return
        mid = (lo + hi) // 2
        nd.left = nd.left or Node(); nd.right = nd.right or Node()
        self._upd(nd.left, lo, mid, l, r); self._upd(nd.right, mid + 1, hi, l, r)
        nd.mx = nd.lazy + max(nd.left.mx, nd.right.mx)
    def book(self, startTime, endTime):
        self._upd(self.root, 0, 10**9, startTime, endTime - 1)
        return self.root.mx

# Bài 5
import heapq
def min_rooms_ans(intervals):
    heap = []
    for s, e in sorted(intervals):
        if heap and heap[0] <= s:
            heapq.heapreplace(heap, e)   # tái dùng phòng sớm trống nhất
        else:
            heapq.heappush(heap, e)
    return len(heap)

rng = random.Random(55)
for _ in range(300):
    evs = []
    a, b = MySegCalendarAns(), MyCalendarThreeBrute()
    for _ in range(rng.randint(1, 20)):
        s = rng.randint(0, 40); e = rng.randint(s + 1, 41); evs.append((s, e))
        assert a.book(s, e) == b.book(s, e)
    assert min_rooms_ans(evs) == max_overlap_ans(evs)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template mảng hiệu trên `dict`:**

```python
delta[s] += 1; delta[e] -= 1
cur = best = 0
for t in sorted(delta):
    cur += delta[t]; best = max(best, cur)
```

**Template Segment Tree động (range add + global max):**

```python
def upd(nd, lo, hi, l, r):
    if r < lo or hi < l: return
    if l <= lo and hi <= r: nd.mx += 1; nd.lazy += 1; return
    mid = (lo + hi) // 2; tạo con nếu thiếu
    upd(nd.left, lo, mid, l, r); upd(nd.right, mid + 1, hi, l, r)
    nd.mx = nd.lazy + max(nd.left.mx, nd.right.mx)
```

**Checklist trước khi submit:**

- [ ] Khoảng nửa mở: chạm mép **không** chồng; segment tree dùng `end - 1`.
- [ ] Gộp Δ theo thời điểm (hoặc `-1` trước `+1`).
- [ ] Trả về max **toàn lịch**, không chỉ quanh sự kiện mới.
- [ ] Segment tree: `mx = lazy + max(con)`.
- [ ] Miền cây phủ `[0, 10^9]`.
- [ ] Tên lớp/method đúng: `MyCalendarThree`, `book(startTime, endTime)`.